# Mencari Missing Value 
(Menggunakan KNN)

K-Nearest Neighbors (KNN) adalah algoritma berbasis "kedekatan". Dalam kasus data yang hilang, prinsipnya adalah: "Jika kita tidak tahu nilai UAS si Ani (ID 2), kita cari orang lain yang nilai UTS dan Nilai Akhirnya paling mirip dengan dia." Nilai UAS dari "tetangga" yang paling mirip itulah yang kita ambil rata-ratanya untuk mengisi kekosongan tersebut.

## Rumus Euclidean
Untuk menentukan tingkat kemiripan (jarak) antar data, kita menggunakan Euclidean Distance. Rumusnya adalah:

$$d(p, q) = \sqrt{\sum_{i=1}^{n} (q_i - p_i)^2}$$

Keterangan: 
- $d(p, q)$ : Jarak antara data target ($p$) dan data tetangga ($q$).
- $q_i, p_i$: Nilai fitur pada kolom ke-$i$ (dalam hal ini kolom Nilai dan UTS).
- $n$: Jumlah fitur yang dibandingkan.

## Pemilihan Fitur
Menggunakan kolom Nilai dan UTS sebagai dasar perhitungan jarak untuk mencari UAS, karena:

- Korelasi Tinggi: Nilai UAS secara logis berkaitan erat dengan nilai UTS dan Nilai Akhir. Jika UTS seseorang tinggi, ada kecenderungan UAS-nya juga tinggi.

- Data Kuantitatif: KNN memerlukan perhitungan matematis. Kolom angka (Nilai, UTS) dapat dihitung jaraknya secara eksak, sedangkan kolom teks (Nama, Gender) tidak bisa dihitung jaraknya tanpa konversi tambahan.

- Menghindari Noise: Kolom seperti ID atau Nama tidak memiliki pengaruh terhadap performa akademik seseorang. Memasukkan ID ke dalam rumus hanya akan merusak akurasi prediksi.



In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Dataset Pelatihan.csv')

df = df.iloc[0:5].copy()

print("Dataset Awal (ID 2 UAS Hilang):")
print(df)

Dataset Awal (ID 2 UAS Hilang):
    ID  Nama  Umur     Gender  Nilai      Matkul    Tanggal   UTS   UAS
0  1.0  Budi  24.0  Laki-laki   85.0       Kimia   9/8/2023  90.0  80.0
1  2.0   Ani  21.0  Perempuan   77.5  Matematika  15/8/2023  75.0   NaN
2  3.0  Joko  20.0  Laki-laki   90.0     Biologi  15/8/2023  85.0  95.0
3  4.0  Siti  21.0  Perempuan   60.0  Matematika   9/8/2023  55.0  65.0
4  5.0  Agus  23.0  Laki-laki   77.5      Fisika   9/8/2023  80.0  75.0


## Perhitungan Jarak
- Jarak ID 2 ke ID 1: 
$$\sqrt{(85 - 77.5)^2 + (90 - 75)^2} = \sqrt{7.5^2 + 15^2} = \sqrt{56.25 + 225} = \mathbf{16.77}$$
- Jarak ID 2 ke ID 3:
$$\sqrt{(90 - 77.5)^2 + (85 - 75)^2} = \sqrt{12.5^2 + 10^2} = \sqrt{156.25 + 100} = \mathbf{16.01}$$
- Jarak ID 2 ke ID 4:
$$\sqrt{(60 - 77.5)^2 + (55 - 75)^2} = \sqrt{(-17.5)^2 + (-20)^2} = \sqrt{306.25 + 400} = \mathbf{26.57}$$
- Jarak ID 2 ke ID 5:
$$\sqrt{(77.5 - 77.5)^2 + (80 - 75)^2} = \sqrt{0^2 + 5^2} = \sqrt{0 + 25} = \mathbf{5.00}$$



In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('Dataset Pelatihan.csv')
df_sample = df[['ID', 'Nilai', 'UTS', 'UAS']].iloc[0:5].copy()

# memisahkan Target dan Tetangga
target = df_sample[df_sample['ID'] == 2].iloc[0]
neighbors_pool = df_sample[df_sample['ID'] != 2].copy()

# hitung jarak
def get_dist(row):
    return np.sqrt((row['Nilai'] - target['Nilai'])**2 + (row['UTS'] - target['UTS'])**2)

neighbors_pool['Jarak'] = neighbors_pool.apply(get_dist, axis=1)

# Mencari rata rata
k_value = 2
top_k = neighbors_pool.sort_values('Jarak').head(k_value)
prediction = top_k['UAS'].mean()

print(f"Jarak tetangga:\n{neighbors_pool[['ID', 'Jarak']]}")
print(f"\nHasil Prediksi Manual UAS ID 2: {prediction}")

Jarak tetangga:
    ID      Jarak
0  1.0  16.770510
2  3.0  16.007811
3  4.0  26.575365
4  5.0   5.000000

Hasil Prediksi Manual UAS ID 2: 85.0


## Memilih Tetangga Terdekat ($K=2$)
Berdasarkan jarak terkecil, dua tetangga terdekat ID 2 adalah:
- ID 5 (Jarak: 5.00)
- ID 3 (Jarak: 16.01)

## Menghitung Rata-rata UAS Tetangga
Ambil nilai UAS dari ID 5 dan ID 3:
- UAS ID 5 = 75
- UAS ID 3 = 95

Prediksi UAS ID 2: 
$\frac{75 + 95}{2} = \mathbf{85}$